# Run Fugue + Nixtla

## Import and Modulize

Accoring to the Ray experiment, we created two steps, generating data and forecasting.

In [ ]:
import argparse
import os
from time import time

import pandas as pd
from statsforecast.utils import generate_series
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, MSTL

In [ ]:
def generate_data(n):
    df = generate_series(n_series=n, seed=1).reset_index()
    return df.assign(unique_id=df.unique_id.astype(int))

# schema: *-y+AutoARIMA:double
def forecast(df:pd.DataFrame) -> pd.DataFrame:
    tdf = df.set_index("unique_id")
    model = StatsForecast(models=[AutoARIMA(season_length = 7)], freq='D',n_jobs=1)
    return model.forecast(h=7,df=tdf).reset_index()

Now test them:

In [ ]:
forecast(generate_data(1))

## Using Fugue Transform

Now rewrite the unit test using Fugue transform function. Also we specify the logical partition key.

This step is still unit testable because the input output are both Pandas Dataframes:

In [ ]:
from fugue import transform

transform(generate_data(2), forecast, partition="unique_id", as_local=True)

### Test on Spark

We only need to add the parameter `engine` to move the compute to the spark cluster on Databricks (`spark` is the SparkSession).

We also add `num` as a partition parameter to control the balance between overhead and load-balance:

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
transform(generate_data(20), forecast, partition={"num":500, "by":"unique_id"}, as_local=True, engine=spark)

### Test on Dask

The only difference is the value of the `engine` parameter:

In [ ]:
%%time
transform(generate_data(10000), forecast, partition={"num":500, "by":"unique_id"}, as_local=True, engine="dask")

In [ ]:
sdf = spark.createDataFrame(generate_data(4))


In [ ]:
import statsforecast.distributed.fugue
model = StatsForecast(models=[AutoARIMA(season_length = 7)], freq='D',n_jobs=1)
model.forecast(h=7, df=sdf).toPandas()

In [ ]:
import fugue.api as fa

with fa.engine_context(spark):
    print(model.forecast(h=7, df=generate_data(4)).toPandas())